In [7]:
### ref/dmrg projections

from pathlib import Path

import numpy as np

from quasisymmetries.block2_qubit_benchmark import (
    run_block2_qubit_reference_dmrg,
)
from quasisymmetries.clifford_symmetry_optimized import Clifford
from quasisymmetries.mps_unitary import (
    mps_configuration_probabilities,
    mps_reduced_density_matrix,
)
from quasisymmetries.save import (
    load_json,
    load_pauli_term_stream,
    load_sparse_qubit_state,
)
from quasisymmetries.sym import hct_mod


# User inputs
K = 10 # number of symmetries
MPS_BOND_DIM = 100

input_dir = Path(
    "saved/results/pyblock2_16_systems/N2_eqm/"
    "orbital_optimization_inputs"
)
manifest = load_json(input_dir / "manifest.json")
n_qubits = int(manifest["n_qubits"])

if not 1 <= K <= n_qubits:
    raise ValueError(f"K must be between 1 and {n_qubits}")

# Load N2 Hamiltonian directly into the packed Pauli-mask representation.
hamiltonian = load_pauli_term_stream(
    input_dir / manifest["hamiltonian_files"]["qubit_json"],
    n_qubits=n_qubits,
)

# CISD state used as the DMRG warm start.
cisd = load_sparse_qubit_state(
    input_dir / manifest["state_files"]["cisd"]
).normalize()

# Find K HCT approximate symmetries.
symmetries, thresholds = hct_mod(
    hamiltonian,
    n_sym=K,
    use_coeffs_eps=True,
    verbose=True,
)

print("\nSelected HCT symmetries:")
for i, (symmetry, threshold) in enumerate(zip(symmetries, thresholds)):
    print(f"{i:2d}: threshold={threshold:.8e}  {symmetry}")

# Construct U such that U S_i U† = Z_i for i = 0, ..., K-1.
clifford = Clifford.from_symmetries(
    symmetries,
    n_qubits=n_qubits,
    symmetry_qubits_first=True,
    synthesis_basis="Z",
    generator_mapping="positive_z",
)

assert clifford.symmetry_qubits == tuple(range(K))

# Rotate the Hamiltonian. The CISD MPS is rotated by the same Clifford inside
# run_block2_qubit_reference_dmrg through unitaries=(clifford,).
rotated_hamiltonian = clifford.transform(hamiltonian)

mps, dmrg_summary = run_block2_qubit_reference_dmrg(
    label=f"N2_HCT_k{K}_chi{MPS_BOND_DIM}",
    hamiltonian=rotated_hamiltonian,
    n_qubits=n_qubits,
    bond_dim=MPS_BOND_DIM,
    sparse_state=(cisd.indices, cisd.coeffs),
    initial_state="cisd",
    unitaries=(clifford,),
    transform_max_bond=MPS_BOND_DIM,
    transform_cutoff=1e-13,
    dmrg_sweeps=100,
    sweep_tolerance=1e-8,
    davidson_threshold=1e-10,
    noises=(1e-4, 1e-4, 1e-5, 1e-5, 1e-6, 1e-6),
    mpo_builder="blocked_sum",
    mpo_cutoff=1e-10,
    sum_mpo_mod=20,
    n_threads=4,
    stack_mem_gb=0.5,
    verbose=False,
    artifact_dir=Path("saved/results/n2_hct_fixed_bond"),
)

print(f"\nDMRG energy: {dmrg_summary['energy']:.12f}")
print(f"Sweeps completed: {dmrg_summary['sweeps_completed']}")
print(f"Sweep converged: {dmrg_summary['sweep_converged']}")

from quasisymmetries.energy_bounds import (
    maximum_omitted_probability_for_block_energy_error,
    symmetry_block_anticommuting_pauli_norms,
    symmetry_block_energy_error_bound,
)
from quasisymmetries.mps_unitary import (
    mps_prefix_configurations_for_probability_mass,
)

CHEMICAL_ACCURACY = 1.6e-3

symmetry_sites = clifford.symmetry_qubits

diagonal_norm, off_diagonal_norm, grouping_info = (
    symmetry_block_anticommuting_pauli_norms(
        rotated_hamiltonian,
        symmetry_sites=symmetry_sites,
        return_diagnostics=True,
    )
)

print(
    "Diagonal ungrouped norm:",
    grouping_info["diagonal_ungrouped_pauli_l1"],
)
print("Diagonal grouped norm:", diagonal_norm)

print(
    "Off-diagonal ungrouped norm:",
    grouping_info["off_diagonal_ungrouped_pauli_l1"],
)
print("Off-diagonal grouped norm:", off_diagonal_norm)

print("Diagonal groups:", len(grouping_info["diagonal_groups"]))
print("Off-diagonal groups:", len(grouping_info["off_diagonal_groups"]))

max_omitted_probability = (
    maximum_omitted_probability_for_block_energy_error(
        diagonal_pauli_l1=diagonal_norm,
        off_diagonal_pauli_l1=off_diagonal_norm,
        energy_tolerance=CHEMICAL_ACCURACY,
    )
)

configurations, search_info = (
    mps_prefix_configurations_for_probability_mass(
        mps,
        n_prefix_sites=K,
        max_omitted_probability=max_omitted_probability,
    )
)

actual_energy_bound = symmetry_block_energy_error_bound(
    omitted_probability=search_info["omitted_probability_bound"],
    diagonal_pauli_l1=diagonal_norm,
    off_diagonal_pauli_l1=off_diagonal_norm,
)

print("Maximum omitted probability:", max_omitted_probability)
print("Actual omitted probability:", search_info["omitted_probability_bound"])
print("Energy-error bound:", actual_energy_bound)
print("Configurations kept:", len(configurations))

1.0 [Z0 Z3 Z4 Z7 Z9 Z11 Z12 Z14 Z16 Z19]  added at threshold 0.0 with metric value  0.0
1.0 [Z0 Z2 Z4 Z6 Z8 Z10 Z12 Z14 Z16 Z18]  added at threshold 0.0 with metric value  0.0
1.0 [Z8 Z9 Z16 Z17]  added at threshold 0.0 with metric value  0.0
1.0 [Z10 Z11 Z14 Z15]  added at threshold 0.0 with metric value  0.0
1.0 [Z0 Z1 Z4 Z5 Z8 Z9 Z10 Z11 Z12 Z13]  added at threshold 0.0 with metric value  0.0
1.0 [Z9 Z17]  added at threshold 0.02666323088461247 with metric value  3.7152706887921907
1.0 [Z11 Z15]  added at threshold 0.02666323088461247 with metric value  3.7152706887921907
1.0 [Z11]  added at threshold 0.08338459974975757 with metric value  5.563994024381873
1.0 [Z10]  added at threshold 0.08338459974975757 with metric value  5.563994024381873
1.0 [Z9]  added at threshold 0.08338459974975757 with metric value  5.563994024381873

Selected HCT symmetries:
 0: threshold=0.00000000e+00  1.0 [Z0 Z3 Z4 Z7 Z9 Z11 Z12 Z14 Z16 Z19]
 1: threshold=0.00000000e+00  1.0 [Z0 Z2 Z4 Z6 Z8 Z10 Z12 Z14

In [15]:
from quasisymmetries.block2_qubit_benchmark import (
    evaluate_qubit_mps_arrays_prefix_projection_energy,
)
from quasisymmetries.energy_bounds import (
    symmetry_block_energy_error_bound,
)

projected_mps, projection_result = (
    evaluate_qubit_mps_arrays_prefix_projection_energy(
        hamiltonian=rotated_hamiltonian,
        tensors=mps,
        retained_configurations=configurations,
        n_prefix_sites=K,
        expected_retained_probability=search_info[
            "retained_probability"
        ],
        probability_tolerance=1e-10,
        mpo_builder="blocked_sum",
        mpo_cutoff=1e-10,
        sum_mpo_mod=20,
        n_threads=4,
        stack_mem_gb=0.5,
    )
)

actual_error = projection_result[
    "actual_projection_energy_error"
]

predicted_bound = symmetry_block_energy_error_bound(
    omitted_probability=projection_result["omitted_probability"],
    diagonal_pauli_l1=diagonal_norm,
    off_diagonal_pauli_l1=off_diagonal_norm,
)

print("Original MPS energy:", projection_result["original_energy"])
print("Projected MPS energy:", projection_result["projected_energy"])
print("Actual projection error:", actual_error)
print("Predicted upper bound:", predicted_bound)
print("Bound / actual error:", predicted_bound / actual_error)

print(
    "Projection probability:",
    projection_result["retained_probability"],
)
print(
    "Input MPS bonds:",
    projection_result["input_mps_bond_dimensions"],
)
print(
    "Projected MPS bonds:",
    projection_result["projected_mps_bond_dimensions"],
)

assert actual_error <= predicted_bound + 1e-10

MPO SIM LEFT BLK ...    0 /   20
MPO SIM LEFT BLK ...    1 /   20
MPO SIM LEFT BLK ...    2 /   20
MPO SIM LEFT BLK ...    3 /   20
MPO SIM LEFT BLK ...    4 /   20
MPO SIM LEFT BLK ...    5 /   20
MPO SIM LEFT BLK ...    6 /   20
MPO SIM LEFT BLK ...    7 /   20
MPO SIM LEFT BLK ...    8 /   20
MPO SIM LEFT BLK ...    9 /   20
MPO SIM LEFT BLK ...   10 /   20
MPO SIM LEFT BLK ...   11 /   20
MPO SIM LEFT BLK ...   12 /   20
MPO SIM LEFT BLK ...   13 /   20
MPO SIM LEFT BLK ...   14 /   20
MPO SIM LEFT BLK ...   15 /   20
MPO SIM LEFT BLK ...   16 /   20
MPO SIM LEFT BLK ...   17 /   20
MPO SIM LEFT BLK ...   18 /   20
MPO SIM LEFT BLK ...   19 /   20
MPO SIM RIGHT BLK ...   19 /   20
MPO SIM RIGHT BLK ...   18 /   20
MPO SIM RIGHT BLK ...   17 /   20
MPO SIM RIGHT BLK ...   16 /   20
MPO SIM RIGHT BLK ...   15 /   20
MPO SIM RIGHT BLK ...   14 /   20
MPO SIM RIGHT BLK ...   13 /   20
MPO SIM RIGHT BLK ...   12 /   20
MPO SIM RIGHT BLK ...   11 /   20
MPO SIM RIGHT BLK ...   10 /   20


In [19]:
rotated_hamiltonian

PauliTermStream(n_qubits=20, terms=(WeightedTerm(mask=(0, 0), abs_coeff=66.20966362068323, term=(), coefficient=(-66.20966362068323+0j)), WeightedTerm(mask=(0, 708553), abs_coeff=9.606898501339346, term=(), coefficient=(9.606898501339346+0j)), WeightedTerm(mask=(2048, 913427), abs_coeff=0.09897377629543279, term=(), coefficient=(-0.09897377629543279+0j)), WeightedTerm(mask=(2048, 473050), abs_coeff=0.09897377629543279, term=(), coefficient=(0.09897377629543279+0j)), WeightedTerm(mask=(32768, 983991), abs_coeff=0.04221129434813627, term=(), coefficient=(0.04221129434813627+0j)), WeightedTerm(mask=(32768, 347262), abs_coeff=0.04221129434813627, term=(), coefficient=(-0.04221129434813627+0j)), WeightedTerm(mask=(0, 611453), abs_coeff=9.606898501339346, term=(), coefficient=(9.606898501339346+0j)), WeightedTerm(mask=(4096, 471002), abs_coeff=0.09897377629543279, term=(), coefficient=(-0.09897377629543279+0j)), WeightedTerm(mask=(4096, 945063), abs_coeff=0.09897377629543279, term=(), coeffi

In [25]:
from __future__ import annotations

import warnings
from typing import Dict, Iterable, List, Optional, Sequence, Set, Tuple

import numpy as np

from openfermion import QubitOperator

from quasisymmetries.bs.utils import (
    PauliMask,
    PauliTermStream,
    as_pauli_term_stream,
    combine_mask,
    symplectic_commutes,
    term_to_masks,
    try_add_to_span,
)
from quasisymmetries.metrics import find_commuting_paulis
from quasisymmetries.state_utils import PauliActionMask, SparseQubitState
def mps_sector_leave_one_out_energies(
    hamiltonian,
    tensors,
    sectors: Sequence[Sequence[int]],
    *,
    n_prefix_sites: Optional[int] = None,
    mpo_builder: str = "blocked_sum",
    mpo_cutoff: float = 1e-10,
    sum_mpo_mod: int = 20,
    n_threads: int = 4,
    stack_mem_gb: float = 0.5,
    scratch=None,
    tag: str = "SECTOR-LOO",
    iprint: int = 0,
):
    """Calculate MPS leave-one-sector-out energies by prefix projection.

    Parameters
    ----------
    hamiltonian, tensors
        Hamiltonian and high-accuracy qubit MPS arrays accepted by
        ``evaluate_qubit_mps_arrays_prefix_projection_energy``.  They must be
        in the frame where the symmetry-sector bits occupy the leading MPS
        sites.
    sectors
        Unique sector labels such as ``[(0, 0, 1), (0, 1, 1), ...]``.  Labels
        are converted to the binary prefix strings required by the projection
        function.  On each run, every listed sector except one is retained.
    n_prefix_sites
        Number of leading symmetry sites.  By default this is inferred from
        the common sector-label length.

    Returns
    -------
    result
        Dictionary containing the original MPS energy, the baseline energy
        projected onto all supplied sectors, projected energies after removing
        each sector, signed leave-one-out energy increases relative to that
        baseline, and the resulting ranking.  Projection tensors, probability
        diagnostics, and energy-bound quantities are intentionally discarded.

    Notes
    -----
    The requested convenience evaluator constructs a fresh Block2 driver and
    Hamiltonian MPO for every omitted sector.  This keeps the wrapper simple
    and stateless, but rebuilding the MPO can dominate runtime for many sectors.
    """
    from quasisymmetries.block2_qubit_benchmark import (
        evaluate_qubit_mps_arrays_prefix_projection_energy,
    )

    sector_tuples = []
    for position, sector in enumerate(sectors):
        try:
            normalized = tuple(int(bit) for bit in sector)
        except (TypeError, ValueError) as exc:
            raise TypeError(
                f"sectors[{position}] must be a sequence of binary integers."
            ) from exc
        if any(bit not in (0, 1) for bit in normalized):
            raise ValueError(f"sectors[{position}] contains a non-binary value.")
        sector_tuples.append(normalized)

    if len(sector_tuples) < 2:
        raise ValueError("At least two sectors are required for leave-one-out.")
    if len(set(sector_tuples)) != len(sector_tuples):
        raise ValueError("sectors must not contain duplicate labels.")

    inferred_prefix_sites = len(sector_tuples[0])
    if inferred_prefix_sites == 0:
        raise ValueError("Sector labels cannot be empty.")
    if any(len(sector) != inferred_prefix_sites for sector in sector_tuples):
        raise ValueError("All sector labels must have the same length.")
    if n_prefix_sites is None:
        n_prefix_sites = inferred_prefix_sites
    n_prefix_sites = int(n_prefix_sites)
    if n_prefix_sites != inferred_prefix_sites:
        raise ValueError(
            "n_prefix_sites must equal the number of bits in each sector label."
        )

    sector_strings = {
        sector: "".join(str(bit) for bit in sector)
        for sector in sector_tuples
    }
    projected_energies = {}
    energy_increases = {}
    _baseline_tensors, baseline_projection = (
        evaluate_qubit_mps_arrays_prefix_projection_energy(
            hamiltonian=hamiltonian,
            tensors=tensors,
            retained_configurations=list(sector_strings.values()),
            n_prefix_sites=n_prefix_sites,
            mpo_builder=mpo_builder,
            mpo_cutoff=mpo_cutoff,
            sum_mpo_mod=sum_mpo_mod,
            n_threads=n_threads,
            stack_mem_gb=stack_mem_gb,
            scratch=scratch,
            tag=f"{tag}-BASELINE",
            iprint=iprint,
        )
    )
    original_energy = float(baseline_projection["original_energy"])
    baseline_energy = float(baseline_projection["projected_energy"])

    for omitted_position, omitted_sector in enumerate(sector_tuples):
        retained_configurations = [
            sector_strings[sector]
            for sector in sector_tuples
            if sector != omitted_sector
        ]
        _projected_tensors, projection = (
            evaluate_qubit_mps_arrays_prefix_projection_energy(
                hamiltonian=hamiltonian,
                tensors=tensors,
                retained_configurations=retained_configurations,
                n_prefix_sites=n_prefix_sites,
                mpo_builder=mpo_builder,
                mpo_cutoff=mpo_cutoff,
                sum_mpo_mod=sum_mpo_mod,
                n_threads=n_threads,
                stack_mem_gb=stack_mem_gb,
                scratch=scratch,
                tag=f"{tag}-DROP-{omitted_position:04d}",
                iprint=iprint,
            )
        )
        run_original_energy = float(projection["original_energy"])
        projected_energy = float(projection["projected_energy"])
        if not np.isclose(
            run_original_energy,
            original_energy,
            rtol=1e-10,
            atol=1e-10,
        ):
            raise RuntimeError(
                "The unprojected MPS energy changed between leave-one-out runs: "
                f"{original_energy} versus {run_original_energy}."
            )
        projected_energies[omitted_sector] = projected_energy
        energy_increases[omitted_sector] = projected_energy - baseline_energy

    ranking = tuple(
        sorted(
            sector_tuples,
            key=lambda sector: (-energy_increases[sector], sector),
        )
    )
    ranking_rows = tuple(
        {
            "rank": rank,
            "sector": sector,
            "sector_string": sector_strings[sector],
            "projected_energy": projected_energies[sector],
            "energy_increase": energy_increases[sector],
        }
        for rank, sector in enumerate(ranking, start=1)
    )
    return {
        "original_energy": original_energy,
        "baseline_energy": baseline_energy,
        "projected_energies": projected_energies,
        "energy_increases": energy_increases,
        "ranking": ranking,
        "ranking_rows": ranking_rows,
    }

ground_truth = mps_sector_leave_one_out_energies(
    hamiltonian=rotated_hamiltonian,
    tensors=mps,
    sectors=analyses[0]['involved_sectors'],
    # Inferred from sector tuple length if omitted:
    n_prefix_sites=K,
    mpo_builder="blocked_sum",
    mpo_cutoff=1e-10,
    sum_mpo_mod=20,
    n_threads=4,
    stack_mem_gb=0.5,
)

print("Original MPS energy:", ground_truth["original_energy"])
print("All-sector baseline:", ground_truth["baseline_energy"])

for row in ground_truth["ranking_rows"]:
    print(
        f"rank={row['rank']:3d}  "
        f"sector={row['sector']}  "
        f"E_without={row['projected_energy']:.12f}  "
        f"dE={row['energy_increase']:+.8e}"
    )

MPO SIM LEFT BLK ...    0 /   20
MPO SIM LEFT BLK ...    1 /   20
MPO SIM LEFT BLK ...    2 /   20
MPO SIM LEFT BLK ...    3 /   20
MPO SIM LEFT BLK ...    4 /   20
MPO SIM LEFT BLK ...    5 /   20
MPO SIM LEFT BLK ...    6 /   20
MPO SIM LEFT BLK ...    7 /   20
MPO SIM LEFT BLK ...    8 /   20
MPO SIM LEFT BLK ...    9 /   20
MPO SIM LEFT BLK ...   10 /   20
MPO SIM LEFT BLK ...   11 /   20
MPO SIM LEFT BLK ...   12 /   20
MPO SIM LEFT BLK ...   13 /   20
MPO SIM LEFT BLK ...   14 /   20
MPO SIM LEFT BLK ...   15 /   20
MPO SIM LEFT BLK ...   16 /   20
MPO SIM LEFT BLK ...   17 /   20
MPO SIM LEFT BLK ...   18 /   20
MPO SIM LEFT BLK ...   19 /   20
MPO SIM RIGHT BLK ...   19 /   20
MPO SIM RIGHT BLK ...   18 /   20
MPO SIM RIGHT BLK ...   17 /   20
MPO SIM RIGHT BLK ...   16 /   20
MPO SIM RIGHT BLK ...   15 /   20
MPO SIM RIGHT BLK ...   14 /   20
MPO SIM RIGHT BLK ...   13 /   20
MPO SIM RIGHT BLK ...   12 /   20
MPO SIM RIGHT BLK ...   11 /   20
MPO SIM RIGHT BLK ...   10 /   20


In [21]:
analyses[0]['involved_sectors']

((1, 1, 0, 0, 0, 1, 0, 0, 0, 0),
 (1, 1, 0, 0, 0, 0, 1, 0, 0, 0),
 (1, 1, 0, 0, 0, 1, 1, 0, 0, 0),
 (1, 1, 0, 0, 0, 0, 0, 1, 0, 0),
 (1, 1, 0, 0, 0, 1, 0, 1, 0, 0),
 (1, 1, 0, 0, 0, 0, 1, 1, 0, 0),
 (1, 1, 0, 0, 0, 1, 1, 1, 0, 0),
 (1, 1, 0, 0, 0, 1, 0, 0, 1, 0),
 (1, 1, 0, 0, 0, 0, 1, 0, 1, 0),
 (1, 1, 0, 0, 0, 1, 1, 0, 1, 0),
 (1, 1, 0, 0, 0, 0, 0, 1, 1, 0),
 (1, 1, 0, 0, 0, 1, 0, 1, 1, 0),
 (1, 1, 0, 0, 0, 0, 1, 1, 1, 0),
 (1, 1, 0, 0, 0, 1, 1, 1, 1, 0),
 (1, 1, 0, 0, 0, 0, 0, 0, 0, 1),
 (1, 1, 0, 0, 0, 1, 0, 0, 0, 1),
 (1, 1, 0, 0, 0, 0, 1, 0, 0, 1),
 (1, 1, 0, 0, 0, 1, 1, 0, 0, 1),
 (1, 1, 0, 0, 0, 0, 0, 1, 0, 1),
 (1, 1, 0, 0, 0, 1, 0, 1, 0, 1),
 (1, 1, 0, 0, 0, 0, 1, 1, 0, 1),
 (1, 1, 0, 0, 0, 1, 1, 1, 0, 1),
 (1, 1, 0, 0, 0, 0, 0, 0, 1, 1),
 (1, 1, 0, 0, 0, 1, 0, 0, 1, 1),
 (1, 1, 0, 0, 0, 0, 1, 0, 1, 1),
 (1, 1, 0, 0, 0, 1, 1, 0, 1, 1),
 (1, 1, 0, 0, 0, 0, 0, 1, 1, 1),
 (1, 1, 0, 0, 0, 1, 0, 1, 1, 1),
 (1, 1, 0, 0, 0, 0, 1, 1, 1, 1),
 (1, 1, 0, 0, 0, 1, 1, 1, 1, 1))

In [ ]:
MPS_BOND_DIM = 100
Maximum omitted probability: 8.3479809731557e-09
Retained probability: 1.000000000000e+00
Omitted probability:  1.098855730890e-15
Energy-error bound:    5.798549828952e-07 Ha
Configurations kept:   30
1100011111  probability=9.217328752145e-01
1100011110  probability=2.021158664742e-02
1100011001  probability=1.766861135646e-02
1100011011  probability=9.948727505257e-03
1100011100  probability=7.405735442827e-03
1100011101  probability=5.304472838943e-03
1100001111  probability=4.335497808900e-03
1100010111  probability=4.161405405104e-03
1100011010  probability=2.761482049564e-03
1100011000  probability=1.835022144641e-03
1100000011  probability=1.214867202748e-03
1100000100  probability=1.213238249452e-03
1100001110  probability=4.516246543040e-04
1100010101  probability=3.189030495243e-04
1100010011  probability=3.189021606467e-04
1100010110  probability=2.065807586626e-04
1100000110  probability=1.643373925263e-04
1100000001  probability=1.627107138610e-04
1100001101  probability=1.448242105821e-04
1100001011  probability=1.448226903411e-04
1100010001  probability=1.333152347317e-04
1100001001  probability=7.395018859485e-05
1100000111  probability=4.078337886078e-05
1100001000  probability=1.506359962280e-05
1100010100  probability=1.219211441297e-05
1100010010  probability=1.219211141637e-05
1100010000  probability=3.464166857009e-06
1100000101  probability=1.626358311992e-06
1100001100  probability=5.927075783535e-07
1100001010  probability=5.926433520749e-07

MPS_BOND_DIM = 10
Maximum omitted probability: 8.3479809731557e-09
Retained probability: 9.999999981280e-01
Omitted probability:  1.872043790983e-09
Energy-error bound:    7.572415313653e-04 Ha
Configurations kept:   27
1100011111  probability=9.253126684382e-01
1100011001  probability=2.135350718367e-02
1100011110  probability=1.981897788208e-02
1100011011  probability=7.976065707467e-03
1100011100  probability=7.734788802063e-03
1100010111  probability=3.499387091539e-03
1100001111  probability=3.472945142545e-03
1100011010  probability=2.502536331876e-03
1100011101  probability=2.447020605625e-03
1100011000  probability=2.206479211895e-03
1100000100  probability=1.274019567245e-03
1100000011  probability=1.181439448964e-03
1100010101  probability=3.192914522617e-04
1100000001  probability=2.684546479627e-04
1100000110  probability=2.287615094270e-04
1100001110  probability=2.040767555647e-04
1100001001  probability=1.099286490047e-04
1100010110  probability=7.221167792821e-05
1100001000  probability=8.200546167564e-06
1100010010  probability=5.878491940010e-06
1100001101  probability=1.676605354477e-06
1100001011  probability=1.516009958742e-06
1100001100  probability=3.455208139801e-08
1100000000  probability=3.454057635765e-08
1100000010  probability=3.451937005657e-08
1100000111  probability=3.151747049580e-08
1100001010  probability=3.123975195152e-08

Reference sector: (1, 1, 0, 0, 0, 1, 1, 1, 1, 1)
Involved sectors: ((1, 1, 0, 0, 0, 1, 0, 0, 0, 0), (1, 1, 0, 0, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 1, 1, 0, 0, 0), (1, 1, 0, 0, 0, 0, 0, 1, 0, 0), (1, 1, 0, 0, 0, 1, 0, 1, 0, 0), (1, 1, 0, 0, 0, 0, 1, 1, 0, 0), (1, 1, 0, 0, 0, 1, 1, 1, 0, 0), (1, 1, 0, 0, 0, 1, 0, 0, 1, 0), (1, 1, 0, 0, 0, 0, 1, 0, 1, 0), (1, 1, 0, 0, 0, 1, 1, 0, 1, 0), (1, 1, 0, 0, 0, 0, 0, 1, 1, 0), (1, 1, 0, 0, 0, 1, 0, 1, 1, 0), (1, 1, 0, 0, 0, 0, 1, 1, 1, 0), (1, 1, 0, 0, 0, 1, 1, 1, 1, 0), (1, 1, 0, 0, 0, 0, 0, 0, 0, 1), (1, 1, 0, 0, 0, 1, 0, 0, 0, 1), (1, 1, 0, 0, 0, 0, 1, 0, 0, 1), (1, 1, 0, 0, 0, 1, 1, 0, 0, 1), (1, 1, 0, 0, 0, 0, 0, 1, 0, 1), (1, 1, 0, 0, 0, 1, 0, 1, 0, 1), (1, 1, 0, 0, 0, 0, 1, 1, 0, 1), (1, 1, 0, 0, 0, 1, 1, 1, 0, 1), (1, 1, 0, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 1, 0, 0, 1, 1), (1, 1, 0, 0, 0, 0, 1, 0, 1, 1), (1, 1, 0, 0, 0, 1, 1, 0, 1, 1), (1, 1, 0, 0, 0, 0, 0, 1, 1, 1), (1, 1, 0, 0, 0, 1, 0, 1, 1, 1), (1, 1, 0, 0, 0, 0, 1, 1, 1, 1), (1, 1, 0, 0, 0, 1, 1, 1, 1, 1))
Full SSPT energy: -107.6799984291435
rank=  1  sector=(1, 1, 0, 0, 0, 1, 1, 1, 1, 0)  dE=+3.76729609e-02  |dE|=3.76729609e-02  infidelity=2.08752483e-02
rank=  2  sector=(1, 1, 0, 0, 0, 1, 1, 0, 0, 1)  dE=+2.97317685e-02  |dE|=2.97317685e-02  infidelity=1.94855407e-02
rank=  3  sector=(1, 1, 0, 0, 0, 1, 1, 0, 1, 1)  dE=+2.74084168e-02  |dE|=2.74084168e-02  infidelity=1.03476290e-02
rank=  4  sector=(1, 1, 0, 0, 0, 1, 1, 1, 0, 1)  dE=+2.34369526e-02  |dE|=2.34369526e-02  infidelity=5.59050800e-03
rank=  5  sector=(1, 1, 0, 0, 0, 1, 1, 1, 0, 0)  dE=+2.16338995e-02  |dE|=2.16338995e-02  infidelity=8.73159472e-03
rank=  6  sector=(1, 1, 0, 0, 0, 1, 1, 0, 1, 0)  dE=+1.86435777e-02  |dE|=1.86435777e-02  infidelity=3.74882914e-03
rank=  7  sector=(1, 1, 0, 0, 0, 1, 1, 0, 0, 0)  dE=+1.47839878e-02  |dE|=1.47839878e-02  infidelity=3.21181379e-03
rank=  8  sector=(1, 1, 0, 0, 0, 0, 1, 1, 1, 1)  dE=+6.74365000e-03  |dE|=6.74365000e-03  infidelity=4.54255271e-03
rank=  9  sector=(1, 1, 0, 0, 0, 1, 0, 1, 1, 1)  dE=+6.36363387e-03  |dE|=6.36363387e-03  infidelity=5.32952682e-03
rank= 10  sector=(1, 1, 0, 0, 0, 1, 0, 1, 1, 0)  dE=+2.62143744e-03  |dE|=2.62143744e-03  infidelity=5.12865126e-04
rank= 11  sector=(1, 1, 0, 0, 0, 0, 1, 0, 1, 1)  dE=+1.17601724e-03  |dE|=1.17601724e-03  infidelity=2.96067534e-04
rank= 12  sector=(1, 1, 0, 0, 0, 0, 1, 1, 0, 1)  dE=+1.16861077e-03  |dE|=1.16861077e-03  infidelity=2.97870196e-04
rank= 13  sector=(1, 1, 0, 0, 0, 0, 1, 1, 1, 0)  dE=+9.62676137e-04  |dE|=9.62676137e-04  infidelity=4.85654009e-04
rank= 14  sector=(1, 1, 0, 0, 0, 0, 0, 0, 1, 1)  dE=+8.50483680e-04  |dE|=8.50483680e-04  infidelity=1.16834443e-03
rank= 15  sector=(1, 1, 0, 0, 0, 0, 0, 1, 0, 0)  dE=+8.41672199e-04  |dE|=8.41672199e-04  infidelity=1.17353663e-03
rank= 16  sector=(1, 1, 0, 0, 0, 0, 1, 0, 0, 1)  dE=+7.27710424e-04  |dE|=7.27710424e-04  infidelity=1.17404645e-04
rank= 17  sector=(1, 1, 0, 0, 0, 1, 0, 1, 0, 1)  dE=+6.70235198e-04  |dE|=6.70235198e-04  infidelity=3.43122033e-04
rank= 18  sector=(1, 1, 0, 0, 0, 1, 0, 0, 1, 1)  dE=+6.63088395e-04  |dE|=6.63088395e-04  infidelity=3.45814086e-04
rank= 19  sector=(1, 1, 0, 0, 0, 1, 0, 0, 0, 1)  dE=+3.07677602e-04  |dE|=3.07677602e-04  infidelity=1.39827344e-04
rank= 20  sector=(1, 1, 0, 0, 0, 0, 0, 1, 1, 0)  dE=-1.42756708e-04  |dE|=1.42756708e-04  infidelity=2.41426299e-04
rank= 21  sector=(1, 1, 0, 0, 0, 0, 0, 0, 0, 1)  dE=-1.39131807e-04  |dE|=1.39131807e-04  infidelity=2.22262716e-04
rank= 22  sector=(1, 1, 0, 0, 0, 0, 0, 1, 1, 1)  dE=+1.19653431e-04  |dE|=1.19653431e-04  infidelity=3.72630166e-05
rank= 23  sector=(1, 1, 0, 0, 0, 0, 1, 0, 0, 0)  dE=+7.08584098e-05  |dE|=7.08584098e-05  infidelity=1.38765950e-05
rank= 24  sector=(1, 1, 0, 0, 0, 1, 0, 1, 0, 0)  dE=+3.51286609e-05  |dE|=3.51286609e-05  infidelity=1.09078209e-05
rank= 25  sector=(1, 1, 0, 0, 0, 1, 0, 0, 1, 0)  dE=+3.48479979e-05  |dE|=3.48479979e-05  infidelity=1.06627763e-05
rank= 26  sector=(1, 1, 0, 0, 0, 1, 0, 0, 0, 0)  dE=+2.76975007e-05  |dE|=2.76975007e-05  infidelity=4.32567507e-06
rank= 27  sector=(1, 1, 0, 0, 0, 0, 0, 1, 0, 1)  dE=+5.12418393e-06  |dE|=5.12418393e-06  infidelity=1.49756239e-06
rank= 28  sector=(1, 1, 0, 0, 0, 0, 1, 0, 1, 0)  dE=+3.63934961e-06  |dE|=3.63934961e-06  infidelity=4.49870895e-07
rank= 29  sector=(1, 1, 0, 0, 0, 0, 1, 1, 0, 0)  dE=+3.07085034e-06  |dE|=3.07085034e-06  infidelity=4.48436285e-07

In [1]:
from pathlib import Path

from quasisymmetries.pt import sspt
from quasisymmetries.save import (
    load_json,
    load_pauli_term_stream,
    load_sparse_qubit_state,
)
from quasisymmetries.sym import hct_mod


input_dir = Path(
    "saved/results/pyblock2_16_systems/N2_eqm/"
    "orbital_optimization_inputs"
)

manifest = load_json(input_dir / "manifest.json")
n_qubits = int(manifest["n_qubits"])

hamiltonian = load_pauli_term_stream(
    input_dir / manifest["hamiltonian_files"]["qubit_json"],
    n_qubits=n_qubits,
)

# Used as a determinant guess: SSPT selects its largest-amplitude determinant.
initial_guess = load_sparse_qubit_state(
    input_dir / manifest["state_files"]["cisd"]
).normalize()

# HCT N/2 symmetries. For this saved N2 Hamiltonian these are I/Z-only,
# so they are already in the computational-basis SSPT frame.
symmetries, thresholds = hct_mod(
    hamiltonian,
    n_sym=n_qubits // 2,
    use_coeffs_eps=True,
    verbose=False,
)

M_ORDER = 4  # maximum number of sector-preserving V0 insertions
N_ORDER = 4  # maximum number of sector-changing V insertions

(
    unperturbed_energy,
    perturbed_energy,
    wavefunction,
    energy_traceback,
    wavefunction_traceback,
) = sspt(
    (M_ORDER, N_ORDER),
    symmetries,
    hamiltonian,
    initial_guess,
    n_qubits=n_qubits,
)

print(f"Z0 reference energy: {unperturbed_energy:.12f}")
print(f"SSPT ground-state energy: {perturbed_energy:.12f}")
print(f"Final wavefunction determinants: {wavefunction.nnz}")

print("\nEnergy corrections:")
for order, correction in sorted(energy_traceback.items()):
    print(f"  {order}: {correction}")

print("\nWavefunction correction support:")
for order, correction in sorted(wavefunction_traceback.items()):
    print(
        f"  {order}: nnz={correction.nnz}, "
        f"norm={correction.norm():.8e}"
    )

1.0 [Z0 Z3 Z4 Z7 Z9 Z11 Z12 Z14 Z16 Z19]  added at threshold 0.0 with metric value  0.0
1.0 [Z0 Z2 Z4 Z6 Z8 Z10 Z12 Z14 Z16 Z18]  added at threshold 0.0 with metric value  0.0
1.0 [Z8 Z9 Z16 Z17]  added at threshold 0.0 with metric value  0.0
1.0 [Z10 Z11 Z14 Z15]  added at threshold 0.0 with metric value  0.0
1.0 [Z0 Z1 Z4 Z5 Z8 Z9 Z10 Z11 Z12 Z13]  added at threshold 0.0 with metric value  0.0
1.0 [Z9 Z17]  added at threshold 0.02666323088461247 with metric value  3.7152706887921907
1.0 [Z11 Z15]  added at threshold 0.02666323088461247 with metric value  3.7152706887921907
1.0 [Z11]  added at threshold 0.08338459974975757 with metric value  5.563994024381873
1.0 [Z10]  added at threshold 0.08338459974975757 with metric value  5.563994024381873
1.0 [Z9]  added at threshold 0.08338459974975757 with metric value  5.563994024381873
Z0 reference energy: -107.496500511798
SSPT ground-state energy: -107.679998429143
Final wavefunction determinants: 1824

Energy corrections:
  (0, 0): -107.4

In [53]:
from quasisymmetries.pt import leave_one_sector_out_sspt

analyses_4 = {}
for M_ORDER in [0, 1, 2, 3]:
    print("\n\nM = ", M_ORDER)
    analysis = leave_one_sector_out_sspt(
        (M_ORDER, 4),
        symmetries,
        hamiltonian,
        initial_guess,
        n_qubits=n_qubits,
    )
    analyses_4[M_ORDER] = analysis

    print("Reference sector:", analysis["reference_sector"])
    print("Involved sectors:", analysis["involved_sectors"])
    print("Full SSPT energy:", analysis["full_energy"])

    for row in analysis["rankings"]:
        print(
            f"rank={row['rank']:3d}  "
            f"sector={row['sector']}  "
            f"dE={row['signed_energy_change']:+.8e}  "
            f"|dE|={row['absolute_energy_change']:.8e}  "
            f"infidelity={row['wavefunction_infidelity']:.8e}"
        )



M =  0
Reference sector: (1, 1, 0, 0, 0, 1, 1, 1, 1, 1)
Involved sectors: ((1, 1, 0, 0, 0, 1, 0, 0, 0, 0), (1, 1, 0, 0, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 1, 1, 0, 0, 0), (1, 1, 0, 0, 0, 0, 0, 1, 0, 0), (1, 1, 0, 0, 0, 1, 0, 1, 0, 0), (1, 1, 0, 0, 0, 0, 1, 1, 0, 0), (1, 1, 0, 0, 0, 1, 1, 1, 0, 0), (1, 1, 0, 0, 0, 1, 0, 0, 1, 0), (1, 1, 0, 0, 0, 0, 1, 0, 1, 0), (1, 1, 0, 0, 0, 1, 1, 0, 1, 0), (1, 1, 0, 0, 0, 0, 0, 1, 1, 0), (1, 1, 0, 0, 0, 1, 0, 1, 1, 0), (1, 1, 0, 0, 0, 0, 1, 1, 1, 0), (1, 1, 0, 0, 0, 1, 1, 1, 1, 0), (1, 1, 0, 0, 0, 0, 0, 0, 0, 1), (1, 1, 0, 0, 0, 1, 0, 0, 0, 1), (1, 1, 0, 0, 0, 0, 1, 0, 0, 1), (1, 1, 0, 0, 0, 1, 1, 0, 0, 1), (1, 1, 0, 0, 0, 0, 0, 1, 0, 1), (1, 1, 0, 0, 0, 1, 0, 1, 0, 1), (1, 1, 0, 0, 0, 0, 1, 1, 0, 1), (1, 1, 0, 0, 0, 1, 1, 1, 0, 1), (1, 1, 0, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 1, 0, 0, 1, 1), (1, 1, 0, 0, 0, 0, 1, 0, 1, 1), (1, 1, 0, 0, 0, 1, 1, 0, 1, 1), (1, 1, 0, 0, 0, 0, 0, 1, 1, 1), (1, 1, 0, 0, 0, 1, 0, 1, 1, 1), (1, 1, 0, 0, 0, 0, 1, 1, 1,

In [67]:
### comparing orderings

import numpy as np


def ndcg(exact_leave_one_out_energies, predicted_order):
    """
    exact_leave_one_out_energies:
        Mapping sector -> exact energy increase caused by removing it.

    predicted_order:
        Sectors ordered from most to least important by SSPT.
    """
    relevance = {
        sector: max(0.0, float(energy))
        for sector, energy in exact_leave_one_out_energies.items()
    }

    exact_order = sorted(
        relevance,
        key=relevance.get,
        reverse=True,
    )

    def dcg(order):
        return sum(
            relevance[sector] / np.log2(rank + 1)
            for rank, sector in enumerate(order, start=1)
        )

    ideal = dcg(exact_order)
    return 1.0 if ideal == 0.0 else dcg(predicted_order) / ideal

def extract_sector_energies_from_ground_truth(ground_truth):
    """
    Includes reference sector (ranked first)

    """

    return {a['sector']: a['energy_increase'] for a in  ground_truth['ranking_rows']}

exact_info = extract_sector_energies_from_ground_truth(ground_truth)

# V order 2
for m in range(4):
    print(ndcg({k: v for k, v in list(exact_info.items())[1:]}, [a['sector'] for a in analyses[m]['rankings']]))

0.9856508855451845
0.9453133449478526
0.9938563573371222
0.9940843528050993


In [79]:
# V order = 4
for m in range(4):
    print(sector_ranking_ndcg(predicted_order= [a['sector'] for a in analyses_4[m]['rankings']], predicted_energy_lowerings= {a['sector']: a['absolute_energy_change'] for a in analyses_4[m]['rankings']}, leave_one_out_energies= {k: v for k, v in list(exact_info.items())[1:]}))

0.9764845093188982
0.9964653768607176
0.9956639643642806
0.9963613255835977


In [ ]:
# V order = 2
for m in range(4):
    print(sector_ranking_ndcg(predicted_order= [a['sector'] for a in analyses[m]['rankings']], predicted_energy_lowerings= {a['sector']: a['absolute_energy_change'] for a in analyses[m]['rankings']}, leave_one_out_energies= {k: v for k, v in list(exact_info.items())[1:]}))

0.984030809451479
0.943708858342517
0.9922368631711543
0.992487903323796


In [ ]:
# V order = 4
for m in range(4):
    print(ndcg({k: v for k, v in list(exact_info.items())[1:]}, [a['sector'] for a in analyses_4[m]['rankings']]))

0.9764845093188982
0.9964653768607176
0.9956639643642806
0.9963613255835977
